# Euclid Q1 Cloud Access Tutorial

## Learning Goals
By the end of this tutorial, you will:

- Learn where Euclid Q1 data are stored in the cloud.
- Retrieve an image cutout from the cloud.
- Retrieve a spectrum from the cloud.

## 1. Introduction
Euclid launched in July 2023 as a European Space Agency mission with NASA involvement.
Euclid Quick Release 1 (Q1) includes imaging, spectroscopy, and catalogs across multiple fields.
IRSA mirrors these data on AWS S3, which this notebook accesses anonymously.

In [ ]:
# Uncomment if needed:
# !pip install s3fs astropy 'astroquery>=0.4.10' matplotlib

import json
import s3fs
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.nddata import Cutout2D
from astropy.wcs import WCS
from astropy.table import Table
from astropy.visualization import ImageNormalize, PercentileInterval, AsinhStretch
from astroquery.ipac.irsa import Irsa
from matplotlib import pyplot as plt

In [ ]:
BUCKET_NAME = 'nasa-irsa-euclid-q1'
s3 = s3fs.S3FileSystem(anon=True)
s3.ls(f'{BUCKET_NAME}/q1')

In [ ]:
s3.ls(f'{BUCKET_NAME}/q1/MER')[:10]

## 2. Spatial Search for MER Mosaics
Use `astroquery` to find cloud paths for a target.

In [ ]:
target_name = 'TYC 4429-1677-1'
coord = SkyCoord.from_name(target_name)
search_radius = 10 * u.arcsec
img_collection = 'euclid_DpdMerBksMosaic'
img_tbl = Irsa.query_sia(pos=(coord, search_radius), collection=img_collection)
len(img_tbl)

In [ ]:
euclid_sci_img_tbl = img_tbl[[
    (row['facility_name'] == 'Euclid') and (row['dataproduct_subtype'] == 'science')
    for row in img_tbl
]]
euclid_sci_img_tbl['instrument_name', 'energy_bandpassname', 'cloud_access']

In [ ]:
def get_s3_fpath(cloud_access):
    cloud_info = json.loads(cloud_access)
    return f"{cloud_info['aws']['bucket_name']}/{cloud_info['aws']['key']}"

def get_filter_name(instrument, bandpass):
    return f"{instrument}_{bandpass}" if instrument != bandpass else instrument

s3_paths = [get_s3_fpath(row['cloud_access']) for row in euclid_sci_img_tbl]
filters = [get_filter_name(row['instrument_name'], row['energy_bandpassname']) for row in euclid_sci_img_tbl]
list(zip(filters, s3_paths))

## 3. Retrieve Image Cutouts
Use FITS lazy loading from S3 and `Cutout2D` to avoid downloading whole mosaics.

In [ ]:
cutout_size = 1 * u.arcmin
cutouts = []
for row in euclid_sci_img_tbl:
    s3_fpath = get_s3_fpath(row['cloud_access'])
    with fits.open(f's3://{s3_fpath}', fsspec_kwargs={'anon': True}) as hdul:
        cutout = Cutout2D(
            hdul[0].section,
            position=coord,
            size=cutout_size,
            wcs=WCS(hdul[0].header),
        )
        cutouts.append(cutout)

fig, axes = plt.subplots(2, 2, figsize=(8, 8), subplot_kw={'projection': cutouts[0].wcs})
for i, ax in enumerate(axes.flat):
    norm = ImageNormalize(cutouts[i].data, interval=PercentileInterval(99), stretch=AsinhStretch())
    ax.imshow(cutouts[i].data, cmap='gray', origin='lower', norm=norm)
    ax.set_xlabel('RA')
    ax.set_ylabel('Dec')
    ax.text(0.95, 0.05, filters[i], color='white', transform=ax.transAxes, ha='right', va='bottom')
plt.tight_layout()

## 4. Find MER Object ID and Retrieve Spectrum
Find a matching object in the MER catalog, then fetch its associated spectrum file and HDU.

In [ ]:
euclid_mer_catalog = 'euclid_q1_mer_catalogue'
euclid_spec_association_catalog = 'euclid.objectid_spectrafile_association_q1'

mer_catalog_tbl = Irsa.query_region(
    coordinates=coord,
    spatial='Cone',
    catalog=euclid_mer_catalog,
    radius=5 * u.arcsec,
)
object_id = int(mer_catalog_tbl['object_id'][0])
object_id

In [ ]:
adql_query = f"SELECT * FROM {euclid_spec_association_catalog} WHERE objectid = {object_id}"
spec_association_tbl = Irsa.query_tap(adql_query).to_table()
spec_association_tbl

In [ ]:
spec_fpath_key = spec_association_tbl['path'][0].replace('api/spectrumdm/convert/euclid/', '').split('?')[0]
object_hdu_idx = int(spec_association_tbl['hdu'][0])

with fits.open(f's3://{BUCKET_NAME}/{spec_fpath_key}', fsspec_kwargs={'anon': True}) as hdul:
    spec_hdu = hdul[object_hdu_idx]
    spec_tbl = Table.read(spec_hdu)
    spec_header = spec_hdu.header

plt.figure(figsize=(8, 4))
plt.plot(spec_tbl['WAVELENGTH'], spec_header['FSCALE'] * spec_tbl['SIGNAL'])
plt.xlabel(spec_tbl['WAVELENGTH'].unit.to_string())
plt.ylabel(spec_tbl['SIGNAL'].unit.to_string())
plt.title(f'Spectrum of {target_name} (Object ID: {object_id})')
plt.tight_layout()

## 5. Use Retrieved Data in This Project
Run the repository utility script to save reusable files under `data/euclid_q1`:

```bash
python scripts/fetch_euclid_q1_data.py --target-name "TYC 4429-1677-1" --out-dir data/euclid_q1
```

Then run model inference on a retrieved cutout image:

```bash
python -m lens_detection.infer --config configs/swin_euclid.yaml --checkpoint runs/swin_euclid_baseline/best.pt --image data/euclid_q1/cutouts/VIS_cutout.png
```